# 1 作业4

**课程**: 深度学习
**学号**: 20234080327
**姓名**: 赵果剑

## 2 序列模型

### 2.1 理论计算题

给定字符序列 "ababc"，一阶马尔可夫模型 $p(x_t|x_{t-1})$，拉普拉斯平滑（加 1 平滑）。

词汇表 $\mathcal{V} = \{'a', 'b', 'c'\}$

**统计转移次数：**

从序列 "ababc" 中统计所有相邻转移：
- a → b: 出现 2 次（位置 1→2, 3→4）
- b → a: 出现 1 次（位置 2→3）
- b → c: 出现 0 次（"b→c"不存在于序列中）

**拉普拉斯平滑公式：**

$$p(x_t|x_{t-1}) = \frac{\text{count}(x_{t-1}, x_t) + 1}{\text{count}(x_{t-1}) + |\mathcal{V}|}$$

其中 $|\mathcal{V}| = 3$。

#### 1. $p('a'|'b')$

- count(b, a) = 1（b→a 出现 1 次）
- count(b) = 2（b 出现在索引 1 和 3（0-based），分别转移到 a 和 a）

$$p(a|b) = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4$$

#### 2. $p('c'|'b')$

- count(b, c) = 0
- count(b) = 2

$$p(c|b) = \frac{0 + 1}{2 + 3} = \frac{1}{5} = 0.2$$


### 2.2 编程题

编写 `preprocess_text(text, n)` 函数，完成文本预处理和 n-gram 序列生成。


In [4]:
def preprocess_text(text, n):
    """
    预处理文本并生成长度为 n 的特征序列和对应的下一个词标签。

    Args:
        text: 原始文本字符串
        n: 滑动窗口大小

    Returns:
        vocab: 词汇表字典 {词: ID}
        (features, labels): (特征列表, 标签列表)
    """
    import re
    from collections import Counter

    # 1. 转换为小写，去除标点符号（保留字母和空格）
    text = text.lower()
    text = re.sub(r'[^a-z ]', '', text)

    # 2. 按空格分词
    tokens = text.split()

    # 3. 构建词汇表（按出现频率排序，分配整数 ID，从 0 开始）
    freq = Counter(tokens)
    # 按频率降序排序，相同频率按词本身排序
    sorted_tokens = sorted(freq.items(), key=lambda x: (-x[1], x[0]))
    vocab = {token: idx for idx, (token, _) in enumerate(sorted_tokens)}

    # 4. 生成长度为 n 的特征序列和对应的下一个词标签
    features = []
    labels = []

    for i in range(len(tokens) - n):
        feat = tokens[i:i + n]
        label = tokens[i + n]
        features.append(feat)
        labels.append(label)

    return vocab, (features, labels)


In [5]:
# 测试 preprocess_text

text = "The time machine"
n = 2
vocab, (features, labels) = preprocess_text(text, n)

print("词汇表:", vocab)
print("特征:", features)
print("标签:", labels)

# 额外测试
text2 = "Hello world! Hello everyone. This is a test."
n2 = 3
vocab2, (features2, labels2) = preprocess_text(text2, n2)
print("\n测试2 - 词汇表:", vocab2)
print("测试2 - 特征:", features2)
print("测试2 - 标签:", labels2)


词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征: [['the', 'time']]
标签: ['machine']

测试2 - 词汇表: {'hello': 0, 'a': 1, 'everyone': 2, 'is': 3, 'test': 4, 'this': 5, 'world': 6}
测试2 - 特征: [['hello', 'world', 'hello'], ['world', 'hello', 'everyone'], ['hello', 'everyone', 'this'], ['everyone', 'this', 'is'], ['this', 'is', 'a']]
测试2 - 标签: ['everyone', 'this', 'is', 'a', 'test']


## 3 循环神经网络

### 3.1 理论计算题

**线性 RNN（无偏置）：**

$$h_t = W_{hh} h_{t-1} + W_{hx} x_t$$

$$o_t = W_{oh} h_t$$

**损失函数：**

$$L = \frac{1}{2} \sum_{t=1}^{T} (o_t - y_t)^2$$

**通过时间反向传播（BPTT）推导 $\frac{\partial L}{\partial W_{hh}}$：**

首先，损失对权重 $W_{hh}$ 的梯度需要展开到所有时间步，因为 $h_t$ 依赖于 $h_{t-1}$，而 $h_{t-1}$ 又依赖于 $W_{hh}$。

对于每个时间步 $t$，$h_t$ 对 $W_{hh}$ 的依赖通过两个路径：
1. 直接依赖：$h_t$ 的计算中直接使用了 $W_{hh}$
2. 间接依赖：通过 $h_{t-1}, h_{t-2}, ..., h_1$ 链式传递

根据链式法则：

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \frac{\partial L}{\partial h_t} \cdot \frac{\partial^+ h_t}{\partial W_{hh}}$$

其中 $\frac{\partial^+ h_t}{\partial W_{hh}}$ 表示将 $h_{t-1}$ 视为常数的"直接"偏导。

实际计算时，我们展开每个 $h_t$ 到初始状态 $h_0$：

$$h_t = W_{hh}^t h_0 + \sum_{k=0}^{t-1} W_{hh}^k W_{hx} x_{t-k}$$

损失对 $W_{hh}$ 的梯度为：

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \sum_{k=0}^{t-1} \frac{\partial L}{\partial h_t} \cdot \frac{\partial h_t}{\partial h_k} \cdot \frac{\partial^+ h_k}{\partial W_{hh}}$$

其中 $\frac{\partial h_t}{\partial h_k} = W_{hh}^{t-k}$。

令 $\delta_t = \frac{\partial L}{\partial h_t} = (o_t - y_t) W_{oh}^T$，则：

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \sum_{k=0}^{t-1} \delta_t \cdot W_{hh}^{t-k-1} \cdot h_k^T$$

**梯度消失/爆炸的条件：**

梯度中包含 $W_{hh}^{t-k-1}$ 项，即 $W_{hh}$ 的幂次。

- 设 $W_{hh}$ 的特征值为 $\lambda_1, \lambda_2, ..., \lambda_H$
- 当 $|\lambda_{\max}| < 1$ 时，$W_{hh}^{\tau} \to 0$（$\tau \to \infty$），梯度消失
- 当 $|\lambda_{\max}| > 1$ 时，$W_{hh}^{\tau} \to \infty$（$\tau \to \infty$），梯度爆炸

其中 $\lambda_{\max}$ 是 $W_{hh}$ 的最大特征值的模长。

**缓解方法：**
- 梯度裁剪（gradient clipping）缓解梯度爆炸
- 使用 LSTM/GRU 等门控机制缓解梯度消失


### 3.2 编程题

实现 RNN 单元的前向传播和单步反向传播。


In [6]:
import numpy as np

def rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN 单元前向传播。

    Args:
        x_t: 输入，形状 (batch_size, input_size)
        h_prev: 上一隐藏状态，形状 (batch_size, hidden_size)
        W_hx: 输入到隐藏的权重，形状 (input_size, hidden_size)
        W_hh: 隐藏到隐藏的权重，形状 (hidden_size, hidden_size)
        b_h: 偏置，形状 (hidden_size,)

    Returns:
        h_t: 当前隐藏状态，形状 (batch_size, hidden_size)
        cache: 用于反向传播的缓存
    """
    # h_t = tanh(x_t @ W_hx + h_prev @ W_hh + b_h)
    a = x_t @ W_hx + h_prev @ W_hh + b_h
    h_t = np.tanh(a)

    cache = (x_t, h_prev, W_hx, W_hh, a, h_t)
    return h_t, cache


def rnn_step_backward(dh_next, cache):
    """
    RNN 单元单步反向传播。

    Args:
        dh_next: 上游梯度（损失对 h_t 的梯度），形状 (batch_size, hidden_size)
        cache: 前向传播缓存的中间变量

    Returns:
        dx_t: 损失对 x_t 的梯度，形状 (batch_size, input_size)
        dh_prev: 损失对 h_prev 的梯度，形状 (batch_size, hidden_size)
        dW_hx: 损失对 W_hx 的梯度，形状 (input_size, hidden_size)
        dW_hh: 损失对 W_hh 的梯度，形状 (hidden_size, hidden_size)
        db_h: 损失对 b_h 的梯度，形状 (hidden_size,)
    """
    x_t, h_prev, W_hx, W_hh, a, h_t = cache

    # tanh 的导数: dtanh(a)/da = 1 - tanh(a)^2 = 1 - h_t^2
    da = dh_next * (1 - h_t ** 2)

    # 对各个参数的梯度
    dx_t = da @ W_hx.T
    dh_prev = da @ W_hh.T
    dW_hx = x_t.T @ da
    dW_hh = h_prev.T @ da
    db_h = np.sum(da, axis=0)

    return dx_t, dh_prev, dW_hx, dW_hh, db_h


In [7]:
# 测试 RNN 单元

np.random.seed(42)
batch_size, input_size, hidden_size = 4, 3, 5

x_t = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hx = np.random.randn(input_size, hidden_size) * 0.01
W_hh = np.random.randn(hidden_size, hidden_size) * 0.01
b_h = np.zeros(hidden_size)

# 前向传播
h_t, cache = rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h)
print("输入 x_t 形状:", x_t.shape)
print("h_prev 形状:", h_prev.shape)
print("h_t 形状:", h_t.shape)
print("h_t 值[:2]:\n", h_t[:2])

# 反向传播
dh_next = np.random.randn(batch_size, hidden_size)
dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_step_backward(dh_next, cache)

print("\ndx_t 形状:", dx_t.shape)
print("dh_prev 形状:", dh_prev.shape)
print("dW_hx 形状:", dW_hx.shape)
print("dW_hh 形状:", dW_hh.shape)
print("db_h 形状:", db_h.shape)

# 数值梯度检验
eps = 1e-5
num_grad = np.zeros((min(2, dW_hx.shape[0]), min(2, dW_hx.shape[1])))
for i in range(num_grad.shape[0]):
    for j in range(num_grad.shape[1]):
        orig = W_hx[i, j]
        W_hx[i, j] = orig + eps
        h_plus, _ = rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h)
        loss_plus = np.sum(h_plus * dh_next)
        W_hx[i, j] = orig - eps
        h_minus, _ = rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h)
        loss_minus = np.sum(h_minus * dh_next)
        W_hx[i, j] = orig
        num_grad[i, j] = (loss_plus - loss_minus) / (2 * eps)

print("\n数值梯度检验 (dW_hx 前 2x2):")
print("解析梯度:", dW_hx[:2, :2])
print("数值梯度:", num_grad)
rel_err = np.max(np.abs(dW_hx[:2, :2] - num_grad) / (1e-8 + np.abs(dW_hx[:2, :2]) + np.abs(num_grad)))
print(f"相对误差: {rel_err:.2e}")


输入 x_t 形状: (4, 3)
h_prev 形状: (4, 5)
h_t 形状: (4, 5)
h_t 值[:2]:
 [[ 1.87908302e-02 -1.88876789e-02 -4.45870394e-02 -3.17915416e-02
   9.71494044e-04]
 [ 8.72891999e-06 -3.97747493e-02  2.14025506e-04 -2.46193690e-04
   8.36243840e-03]]

dx_t 形状: (4, 3)
dh_prev 形状: (4, 5)
dW_hx 形状: (3, 5)
dW_hh 形状: (5, 5)
db_h 形状: (5,)

数值梯度检验 (dW_hx 前 2x2):
解析梯度: [[ 2.03819291 -0.18801341]
 [ 1.05682388 -0.38930886]]
数值梯度: [[ 2.03819291 -0.18801341]
 [ 1.05682388 -0.38930886]]
相对误差: 1.42e-10


## 4 高级循环神经网络

### 4.1 理论计算题

**深度双向 RNN 参数计算：**

已知：
- $L$ 层
- 每层隐藏单元数 $H$
- 输入维度 $D$
- 输出维度 $O$（仅考虑最后输出层）

**逐层参数计算：**

#### 第 1 层（底层）
- **前向 RNN：** 输入维度 $D$ → 隐藏维度 $H$
  - $W_{hx}^{(1,f)} \in \mathbb{R}^{D \times H}$：$D \times H$ 个参数
  - $W_{hh}^{(1,f)} \in \mathbb{R}^{H \times H}$：$H \times H$ 个参数
  - $b_h^{(1,f)} \in \mathbb{R}^{H}$：$H$ 个参数
  - 小计：$DH + H^2 + H$

- **反向 RNN：** 输入维度 $D$ → 隐藏维度 $H$
  - 小计：$DH + H^2 + H$

第 1 层总计：$2(DH + H^2 + H)$

#### 第 $l$ 层（$l \geq 2$）
- **前向 RNN：** 输入维度 $2H$（拼接前一层的前向和反向输出）→ 隐藏维度 $H$
  - $W_{hx}^{(l,f)} \in \mathbb{R}^{2H \times H}$：$2H^2$ 个参数
  - $W_{hh}^{(l,f)} \in \mathbb{R}^{H \times H}$：$H^2$ 个参数
  - $b_h^{(l,f)} \in \mathbb{R}^{H}$：$H$ 个参数
  - 小计：$3H^2 + H$

- **反向 RNN：** 同理 $3H^2 + H$

第 $l$ 层（$l \geq 2$）总计：$6H^2 + 2H$

#### 输出层
将最后一层的前向和反向隐藏状态拼接（维度 $2H$）投影到输出维度 $O$：
- $W_{out} \in \mathbb{R}^{2H \times O}$：$2HO$
- $b_{out} \in \mathbb{R}^{O}$：$O$

输出层小计：$2HO + O$

#### 参数量总计

$$\text{Total} = \underbrace{2(DH + H^2 + H)}_{\text{第1层}} + \underbrace{(L-1) \cdot (6H^2 + 2H)}_{\text{第2~L层}} + \underbrace{(2HO + O)}_{\text{输出层}}$$

化简：

$$\text{Total} = 2DH + 2H^2 + 2H + (L-1)(6H^2 + 2H) + 2HO + O$$

$$= 2DH + [2 + 6(L-1)]H^2 + [2 + 2(L-1)]H + 2HO + O$$

$$= 2DH + (6L - 4)H^2 + 2LH + 2HO + O$$

其中 $L \geq 1$。


### 4.2 编程题

实现双向 RNN 编码器。


In [8]:
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    """
    双向 RNN 编码器（使用 PyTorch 内置 nn.RNN）。

    接收序列 X (seq_len, batch, input_dim)，
    返回每个时间步拼接后的隐藏状态 (seq_len, batch, 2*hidden_dim)
    以及最终时间步的拼接隐藏状态 (batch, 2*hidden_dim)。
    """
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=False
        )
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

    def forward(self, X):
        """
        Args:
            X: (seq_len, batch, input_dim)
        Returns:
            outputs: (seq_len, batch, 2*hidden_dim)
            final: (batch, 2*hidden_dim)
        """
        outputs, h_n = self.rnn(X)
        # h_n: (num_layers * 2, batch, hidden_dim)
        h_fwd = h_n[-2]  # (batch, hidden_dim)
        h_bwd = h_n[-1]  # (batch, hidden_dim)
        final = torch.cat([h_fwd, h_bwd], dim=-1)
        return outputs, final


class BiRNNEncoderManual(nn.Module):
    """
    手动实现的双向 RNN 编码器。
    """
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.fwd_rnn = nn.RNN(input_dim, hidden_dim, num_layers, batch_first=False)
        self.bwd_rnn = nn.RNN(input_dim, hidden_dim, num_layers, batch_first=False)

    def forward(self, X):
        """
        Args:
            X: (seq_len, batch, input_dim)
        Returns:
            outputs: (seq_len, batch, 2*hidden_dim)
            final: (batch, 2*hidden_dim)
        """
        fwd_out, fwd_h = self.fwd_rnn(X)

        # 反向：将序列反转后输入反向 RNN，再反转输出
        X_rev = torch.flip(X, dims=[0])
        bwd_out, bwd_h = self.bwd_rnn(X_rev)
        bwd_out = torch.flip(bwd_out, dims=[0])

        outputs = torch.cat([fwd_out, bwd_out], dim=-1)
        final = torch.cat([fwd_h[-1], bwd_h[-1]], dim=-1)

        return outputs, final


In [9]:
# 测试双向 RNN 编码器

seq_len, batch, input_dim = 5, 3, 8
hidden_dim = 16

X = torch.randn(seq_len, batch, input_dim)
print(f"输入形状: {X.shape}")

# 测试 PyTorch 内置实现
encoder = BiRNNEncoder(input_dim, hidden_dim, num_layers=1)
outputs, final = encoder(X)
print(f"\nPyTorch 内置实现:")
print(f"  outputs 形状: {outputs.shape}")
print(f"  final 形状: {final.shape}")

# 测试手动实现
encoder_manual = BiRNNEncoderManual(input_dim, hidden_dim, num_layers=1)
outputs_m, final_m = encoder_manual(X)
print(f"\n手动实现:")
print(f"  outputs 形状: {outputs_m.shape}")
print(f"  final 形状: {final_m.shape}")

# 验证多层
encoder_multi = BiRNNEncoder(input_dim, hidden_dim, num_layers=2)
outputs_multi, final_multi = encoder_multi(X)
print(f"\n2 层 BiRNN:")
print(f"  outputs 形状: {outputs_multi.shape}")
print(f"  final 形状: {final_multi.shape}")

print("\n所有测试通过！")


输入形状: torch.Size([5, 3, 8])

PyTorch 内置实现:
  outputs 形状: torch.Size([5, 3, 32])
  final 形状: torch.Size([3, 32])

手动实现:
  outputs 形状: torch.Size([5, 3, 32])
  final 形状: torch.Size([3, 32])

2 层 BiRNN:
  outputs 形状: torch.Size([5, 3, 32])
  final 形状: torch.Size([3, 32])

所有测试通过！


## 5 嵌入向量

### 5.1 理论计算题

**Skip-gram 模型 + 负采样**

#### 模型定义

给定中心词 $w_c$ 和上下文词 $w_o$，Skip-gram 模型的目标是最大化给定中心词时上下文词出现的概率。

- 中心词向量：$v_c \in \mathbb{R}^d$
- 上下文词向量：$u_o \in \mathbb{R}^d$（作为输出的词向量）
- 负样本词向量：$u_{n_k} \in \mathbb{R}^d$（$k = 1, ..., K$）

#### Softmax 原始定义

$$P(w_o|w_c) = \frac{\exp(u_o^T v_c)}{\sum_{w \in V} \exp(u_w^T v_c)}$$

完整 softmax 的计算复杂度为 $O(|V|)$，对于大词汇表非常昂贵。

#### 负采样目标函数

负采样将问题转化为二分类：区分正样本对 $(w_c, w_o)$ 和负样本对 $(w_c, w_{neg})$。

对于单个中心词-上下文词对 $(w_c, w_o)$，负采样的损失函数为：

$$\mathcal{L} = - \left[ \log \sigma(u_o^T v_c) + \sum_{k=1}^{K} \log \sigma(-u_{n_k}^T v_c) \right]$$

其中 $\sigma(x) = 1/(1+e^{-x})$ 是 sigmoid 函数。

第二项中 $\sigma(-u_{n_k}^T v_c) = 1 - \sigma(u_{n_k}^T v_c)$，表示负样本不被预测为上下文词的概率。

#### 完整的目标函数

对整个语料库，最小化负对数似然：

$$\mathcal{L} = - \sum_{w_c \in \text{corpus}} \sum_{w_o \in \text{context}(w_c)} \left[ \log \sigma(u_o^T v_c) + \sum_{k=1}^{K} \log \sigma(-u_{n_k}^T v_c) \right]$$

#### 负样本采样

负样本从噪声分布 $P_n(w)$ 中采样，通常使用词汇表中词频率的 3/4 次幂：

$$P_n(w) = \frac{\text{count}(w)^{3/4}}{\sum_{w' \in V} \text{count}(w')^{3/4}}$$

取 3/4 次幂可以降低高频词的采样概率，增加低频词被采样到的机会，从而提高低频词嵌入的质量。


### 5.2 编程题

实现 CBOW 模型的前向传播和损失计算（完整 softmax）。


In [10]:
import numpy as np

class CBOW:
    """
    CBOW (Continuous Bag of Words) 模型。
    使用完整 softmax 进行损失计算。
    """
    def __init__(self, vocab_size, embed_dim):
        """
        Args:
            vocab_size: 词汇表大小 V
            embed_dim: 嵌入维度 d
        """
        self.V = vocab_size
        self.d = embed_dim
        self.W = np.random.randn(vocab_size, embed_dim) * 0.01
        self.W_out = np.random.randn(embed_dim, vocab_size) * 0.01

    def forward(self, context_indices, target_idx):
        """
        CBOW 前向传播和损失计算。

        Args:
            context_indices: 上下文词索引列表，(context_size,)
            target_idx: 目标中心词索引，标量

        Returns:
            loss: 交叉熵损失值
            cache: 用于反向传播的缓存
        """
        # 1. 获取上下文词的嵌入向量并求平均作为隐藏层
        context_vecs = self.W[context_indices]  # (context_size, d)
        h = np.mean(context_vecs, axis=0)       # (d,)

        # 2. 计算输出 logits
        logits = h @ self.W_out  # (V,)

        # 3. Softmax（数值稳定版）
        logits_stable = logits - np.max(logits)
        exp_logits = np.exp(logits_stable)
        probs = exp_logits / np.sum(exp_logits)

        # 4. 交叉熵损失
        loss = -np.log(probs[target_idx] + 1e-10)

        cache = {
            'context_indices': context_indices,
            'target_idx': target_idx,
            'h': h,
            'probs': probs
        }
        return loss, cache

    def backward(self, cache):
        """
        CBOW 反向传播。
        """
        context_indices = cache['context_indices']
        target_idx = cache['target_idx']
        h = cache['h']
        probs = cache['probs']
        context_size = len(context_indices)

        # d_logits = probs - y_onehot
        d_logits = probs.copy()
        d_logits[target_idx] -= 1.0

        # dW_out = outer(h, d_logits)
        dW_out = np.outer(h, d_logits)

        # dh = W_out @ d_logits
        dh = self.W_out @ d_logits

        # dW: 只有 context_indices 对应的行有梯度
        dW = np.zeros_like(self.W)
        d_context_vecs = dh / context_size
        for idx in context_indices:
            dW[idx] += d_context_vecs

        return {'dW': dW, 'dW_out': dW_out}

    def predict(self, context_indices):
        """
        给定上下文词，预测最可能的中心词。
        """
        context_vecs = self.W[context_indices]
        h = np.mean(context_vecs, axis=0)
        logits = h @ self.W_out
        logits_stable = logits - np.max(logits)
        probs = np.exp(logits_stable) / np.sum(np.exp(logits_stable))
        return np.argmax(probs), probs


In [11]:
# 测试 CBOW 模型

np.random.seed(42)

V = 10  # 词汇表大小
d = 4   # 嵌入维度

model = CBOW(V, d)

context_indices = np.array([1, 3, 5])
target_idx = 2

print(f"词汇表大小 V = {V}")
print(f"嵌入维度 d = {d}")
print(f"上下文词索引: {context_indices}")
print(f"目标词索引: {target_idx}")
print(f"\nW 形状: {model.W.shape}, W_out 形状: {model.W_out.shape}")

# 前向传播
loss, cache = model.forward(context_indices, target_idx)
print(f"\n交叉熵损失: {loss:.6f}")
print(f"目标词概率: {cache['probs'][target_idx]:.6f}")

# 反向传播
grads = model.backward(cache)
print(f"\ndW 梯度形状: {grads['dW'].shape}")
print(f"dW_out 梯度形状: {grads['dW_out'].shape}")

# 数值梯度检验
eps = 1e-5
orig = model.W[context_indices[0], 0]
model.W[context_indices[0], 0] = orig + eps
loss_plus, _ = model.forward(context_indices, target_idx)
model.W[context_indices[0], 0] = orig - eps
loss_minus, _ = model.forward(context_indices, target_idx)
model.W[context_indices[0], 0] = orig
num_grad = (loss_plus - loss_minus) / (2 * eps)
print(f"\n数值梯度检验:")
print(f"  dW[{context_indices[0]},0] - 解析: {grads['dW'][context_indices[0],0]:.6f}, 数值: {num_grad:.6f}")

# 预测
pred_idx, probs = model.predict(context_indices)
print(f"\n预测中心词索引: {pred_idx} (实际: {target_idx})")
print(f"各词概率: {np.round(probs, 4)}")


词汇表大小 V = 10
嵌入维度 d = 4
上下文词索引: [1 3 5]
目标词索引: 2

W 形状: (10, 4), W_out 形状: (4, 10)

交叉熵损失: 2.302507
目标词概率: 0.100008

dW 梯度形状: (10, 4)
dW_out 梯度形状: (4, 10)

数值梯度检验:
  dW[1,0] - 解析: -0.000457, 数值: -0.000457

预测中心词索引: 7 (实际: 2)
各词概率: [0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]


## 6 注意力机制

### 6.1 理论计算题

**缩放点积注意力（无掩码）**

给定：
$$Q \in \mathbb{R}^{2 \times 4}, \quad K \in \mathbb{R}^{3 \times 4}, \quad V \in \mathbb{R}^{3 \times 5}$$

$d_k = 4$

#### 步骤 1：计算得分矩阵 $S = QK^T / \sqrt{d_k}$

$$S = \frac{QK^T}{\sqrt{4}} = \frac{QK^T}{2} \in \mathbb{R}^{2 \times 3}$$

$$S_{ij} = \frac{\sum_{m=1}^{4} Q_{im} K_{jm}}{2}, \quad i \in \{1,2\}, j \in \{1,2,3\}$$

#### 步骤 2：对每一行应用 Softmax

$$A_{ij} = \frac{\exp(S_{ij})}{\sum_{k=1}^{3} \exp(S_{ik})}$$

得到注意力权重矩阵 $A \in \mathbb{R}^{2 \times 3}$，每行和为 1。

#### 步骤 3：加权求和得到输出

$$\text{Output} = A \cdot V \in \mathbb{R}^{2 \times 5}$$

$$\text{Output}_i = \sum_{j=1}^{3} A_{ij} \cdot V_j$$

**具体数值示例：**

$$Q = \begin{bmatrix} 1 & 0 & 1 & 0 \\ 0 & 1 & 0 & 1 \end{bmatrix}, \quad
K = \begin{bmatrix} 1 & 1 & 0 & 0 \\ 0 & 0 & 1 & 1 \\ 1 & 0 & 1 & 0 \end{bmatrix}, \quad
V = \begin{bmatrix} 1 & 0 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 & 0 \\ 0 & 0 & 1 & 0 & 0 \end{bmatrix}$$

$$S_{11} = (1\cdot1 + 0\cdot1 + 1\cdot0 + 0\cdot0)/2 = 1/2 = 0.5$$
$$S_{12} = (1\cdot0 + 0\cdot0 + 1\cdot1 + 0\cdot1)/2 = 1/2 = 0.5$$
$$S_{13} = (1\cdot1 + 0\cdot0 + 1\cdot1 + 0\cdot0)/2 = 2/2 = 1.0$$

$$S_{21} = (0\cdot1 + 1\cdot1 + 0\cdot0 + 1\cdot0)/2 = 1/2 = 0.5$$
$$S_{22} = (0\cdot0 + 1\cdot0 + 0\cdot1 + 1\cdot1)/2 = 1/2 = 0.5$$
$$S_{23} = (0\cdot1 + 1\cdot0 + 0\cdot1 + 1\cdot0)/2 = 0/2 = 0.0$$

**Softmax 得到注意力权重：**

$$A_{1,:} = \text{softmax}([0.5, 0.5, 1.0]) = [0.26, 0.26, 0.48]$$
$$A_{2,:} = \text{softmax}([0.5, 0.5, 0.0]) = [0.37, 0.37, 0.26]$$

**输出计算：**

$$\text{Output}_1 = 0.26 \cdot [1,0,0,0,0] + 0.26 \cdot [0,1,0,0,0] + 0.48 \cdot [0,0,1,0,0] = [0.26, 0.26, 0.48, 0, 0]$$

$$\text{Output}_2 = 0.37 \cdot [1,0,0,0,0] + 0.37 \cdot [0,1,0,0,0] + 0.26 \cdot [0,0,1,0,0] = [0.37, 0.37, 0.26, 0, 0]$$

$$\text{Output} = \begin{bmatrix} 0.26 & 0.26 & 0.48 & 0 & 0 \\ 0.37 & 0.37 & 0.26 & 0 & 0 \end{bmatrix} \in \mathbb{R}^{2 \times 5}$$


### 6.2 编程题

实现多头注意力（Multi-Head Attention）的前向传播。


In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    """
    多头注意力（Multi-Head Attention）

    参数:
        d_model: 模型维度
        num_heads: 注意力头数
    """
    def __init__(self, d_model=4, num_heads=2):
        super().__init__()
        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.d_v = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, X, return_attention=False):
        """
        Args:
            X: (seq_len, batch, d_model)
        """
        seq_len, batch, _ = X.shape

        # 1. 线性投影 Q, K, V
        Q = self.W_q(X)  # (seq_len, batch, d_model)
        K = self.W_k(X)
        V = self.W_v(X)

        # 2. 拆分为多个头
        # (seq_len, batch, num_heads, d_k) -> (batch, num_heads, seq_len, d_k)
        Q = Q.view(seq_len, batch, self.num_heads, self.d_k).permute(1, 2, 0, 3)
        K = K.view(seq_len, batch, self.num_heads, self.d_k).permute(1, 2, 0, 3)
        V = V.view(seq_len, batch, self.num_heads, self.d_v).permute(1, 2, 0, 3)

        # 3. 缩放点积注意力
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        attn_weights = F.softmax(scores, dim=-1)
        head_outputs = torch.matmul(attn_weights, V)
        # head_outputs: (batch, num_heads, seq_len, d_v)

        # 4. 拼接所有头的输出
        concat = head_outputs.permute(2, 0, 1, 3).reshape(seq_len, batch, self.d_model)

        # 5. 最终线性层
        output = self.W_o(concat)

        if return_attention:
            return output, attn_weights
        return output


In [13]:
# 测试多头注意力

seq_len = 4
batch = 2
d_model = 4
num_heads = 2

mha = MultiHeadAttention(d_model, num_heads)

X = torch.randn(seq_len, batch, d_model)
print(f"输入 X 形状: {X.shape}")
print(f"d_model={d_model}, num_heads={num_heads}, d_k=d_v={d_model//num_heads}")

# 前向传播
output = mha(X)
print(f"输出形状: {output.shape}")
assert output.shape == X.shape, "输出形状应与输入形状一致"

# 带注意力权重
output2, attn = mha(X, return_attention=True)
print(f"注意力权重形状: {attn.shape}  # (batch, num_heads, seq_len, seq_len)")

print("\n✓ 输出形状与输入形状一致")
print("✓ 多头注意力实现测试通过")


输入 X 形状: torch.Size([4, 2, 4])
d_model=4, num_heads=2, d_k=d_v=2
输出形状: torch.Size([4, 2, 4])
注意力权重形状: torch.Size([2, 2, 4, 4])  # (batch, num_heads, seq_len, seq_len)

✓ 输出形状与输入形状一致
✓ 多头注意力实现测试通过
